In [14]:
from typing import TypedDict, Dict, Any
from langgraph.graph import StateGraph, START, END

In [15]:
# Layer 0: Regulatory Intake sequential workflow (LangGraph)

In [16]:
class RegulatoryIntakeState(TypedDict, total=False):
    RBI_Circular: str
    Email: str
    ManualUpload: str
    source: str
    raw_text: str
    extracted_text: str
    metadata: Dict[str, Any]
    document_id: str
    status: str

In [17]:
def normalize_input(state: RegulatoryIntakeState) -> dict:
    if state.get("RBI_Circular"):
        return {
            "source": "RBI_Circular",
            "raw_text": state["RBI_Circular"],
            "status": "circular_received",
        }
    if state.get("Email"):
        return {
            "source": "Email",
            "raw_text": state["Email"],
            "status": "email_received",
        }
    if state.get("ManualUpload"):
        return {
            "source": "ManualUpload",
            "raw_text": state["ManualUpload"],
            "status": "upload_received",
        }
    return {"status": "no_input_found"}

In [18]:
def ocr_parsing(state: RegulatoryIntakeState) -> dict:
    text = state.get("raw_text", "")
    if state.get("source") == "ManualUpload":
        extracted_text = f"OCR extracted text from upload: {text}"
    else:
        extracted_text = text
    return {
        "extracted_text": extracted_text,
        "status": "ocr_parsed",
    }


def metadata_extraction(state: RegulatoryIntakeState) -> dict:
    text = state.get("extracted_text", "")
    metadata = {
        "document_type": "RBI Circular",
        "source": state.get("source", ""),
        "summary": text[:150],
        "length": len(text),
    }
    return {
        "metadata": metadata,
        "status": "metadata_extracted",
    }


def document_store(state: RegulatoryIntakeState) -> dict:
    # Replace with DB/object-store write in production
    return {
        "document_id": "DOC-001",
        "status": "stored",
    }

In [19]:
graph = StateGraph(RegulatoryIntakeState)

graph.add_node("normalize_input", normalize_input)
graph.add_node("ocr_parsing", ocr_parsing)
graph.add_node("metadata_extraction", metadata_extraction)
graph.add_node("document_store", document_store)

graph.add_edge(START, "normalize_input")
graph.add_edge("normalize_input", "ocr_parsing")
graph.add_edge("ocr_parsing", "metadata_extraction")
graph.add_edge("metadata_extraction", "document_store")
graph.add_edge("document_store", END)

app = graph.compile()

sample_input = {
    "ManualUpload": "RBI circular scanned PDF content regarding KYC updates.",
}

result = app.invoke(sample_input)
result

{'ManualUpload': 'RBI circular scanned PDF content regarding KYC updates.',
 'source': 'ManualUpload',
 'raw_text': 'RBI circular scanned PDF content regarding KYC updates.',
 'extracted_text': 'OCR extracted text from upload: RBI circular scanned PDF content regarding KYC updates.',
 'metadata': {'document_type': 'RBI Circular',
  'source': 'ManualUpload',
  'summary': 'OCR extracted text from upload: RBI circular scanned PDF content regarding KYC updates.',
  'length': 87},
 'document_id': 'DOC-001',
 'status': 'stored'}